# Notebook 2: Data Preprocessing
## AI-Powered Job Market Intelligence — Sri Lanka

### What we do here:
- Load raw data from Notebook 1
- Clean and normalize text
- Remove noise and stopwords
- Save cleaned data for Notebook 3

Input  → data/raw/jobs_raw.csv
Output → data/processed/jobs_cleaned.csv

In [2]:
import subprocess
import sys

packages = ['nltk', 'tqdm']

for package in packages:
    print(f"Installing {package}...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])
    print(f"✅ {package} installed!\n")

print("All done!")

Installing nltk...
✅ nltk installed!

Installing tqdm...
✅ tqdm installed!

All done!


In [3]:
# ── IMPORTS ──────────────────────────────────

import pandas as pd        # Data manipulation
import numpy as np         # Numerical operations
import re                  # Regular expressions - for pattern matching in text
import os                  # File and folder operations
import nltk                # Natural Language Toolkit
from tqdm.notebook import tqdm  # Progress bars

# Download required NLTK data
nltk.download('stopwords')        # Common words to ignore: 'the', 'a', 'is'
nltk.download('punkt')            # Word and sentence tokenizer
nltk.download('wordnet')          # Word meanings for lemmatization
nltk.download('punkt_tab')        # Updated tokenizer data

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

print("✅ All libraries imported!")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lap.lk\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\lap.lk\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\lap.lk\AppData\Roaming\nltk_data...
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\lap.lk\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


✅ All libraries imported!


In [4]:
# ── SET PATHS ────────────────────────────────

BASE_PATH            = 'D:/Sinali/projects/job_market_intelligence_platform'
RAW_DATA_PATH        = f'{BASE_PATH}/data/raw'
PROCESSED_DATA_PATH  = f'{BASE_PATH}/data/processed'
OUTPUT_PATH          = f'{BASE_PATH}/outputs'

# Create folders if not exist
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)

# ── LOAD RAW DATA ────────────────────────────

print("📂 Loading raw data...")

df = pd.read_csv(f'{RAW_DATA_PATH}/jobs_raw.csv')

print(f"✅ Data loaded!")
print(f"📊 Rows    : {len(df)}")
print(f"📊 Columns : {len(df.columns)}")
print(f"\n📋 First 2 rows:")
df.head(2)

📂 Loading raw data...
✅ Data loaded!
📊 Rows    : 1615940
📊 Columns : 12

📋 First 2 rows:


,title,description,skills,salary,experience,qualification,location,country,work_type,company,date_posted,role
0,Digital Marketing Specialist,Social Media Managers oversee an organizations...,"Social media platforms (e.g., Facebook, Twitte...",$59K-$99K,5 to 15 Years,M.Tech,Douglas,Isle of Man,Intern,Icahn Enterprises,2022-04-24,Social Media Manager
1,Web Developer,Frontend Web Developers design and implement u...,"HTML, CSS, JavaScript Frontend frameworks (e.g...",$56K-$116K,2 to 12 Years,BCA,Ashgabat,Turkmenistan,Intern,PNC Financial Services Group,2022-12-19,Frontend Web Developer


In [5]:
# ── TEXT CLEANING FUNCTIONS ──────────────────

def remove_special_characters(text):
    """
    Removes special characters from text
    Example: 'Python 3.9! (required)' → 'Python 39 required'
    """
    if pd.isna(text):
        return ''
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', str(text))
    return text


def to_lowercase(text):
    """
    Converts text to lowercase
    Example: 'Python DEVELOPER' → 'python developer'
    """
    if pd.isna(text):
        return ''
    return str(text).lower()


def remove_extra_whitespace(text):
    """
    Removes extra spaces and newlines
    Example: 'python   developer' → 'python developer'
    """
    if pd.isna(text):
        return ''
    return ' '.join(str(text).split())


def remove_stopwords(text):
    """
    Removes common words that carry no meaning
    Example: 'we are looking for a developer' → 'looking developer'
    """
    if pd.isna(text) or text == '':
        return ''

    # Standard English stopwords
    stop_words = set(stopwords.words('english'))

    # Extra job posting words to ignore
    custom_stopwords = {
        'looking', 'seeking', 'candidate', 'experience', 'required',
        'ability', 'strong', 'excellent', 'good', 'knowledge',
        'join', 'team', 'company', 'opportunity', 'role', 'position',
        'responsibilities', 'qualifications', 'requirements',
        'apply', 'send', 'cv', 'resume', 'years', 'year', 'minimum'
    }
    stop_words.update(custom_stopwords)

    tokens = word_tokenize(str(text))
    filtered = [word for word in tokens if word not in stop_words and len(word) > 2]

    return ' '.join(filtered)


def lemmatize_text(text):
    """
    Reduces words to their base form
    Example: 'developing' → 'develop'
             'technologies' → 'technology'
    """
    if pd.isna(text) or text == '':
        return ''

    lemmatizer = WordNetLemmatizer()
    tokens = word_tokenize(str(text))
    lemmatized = [lemmatizer.lemmatize(word) for word in tokens]

    return ' '.join(lemmatized)


def clean_text(text):
    """
    Master function — runs all steps in order
    """
    text = remove_special_characters(text)
    text = to_lowercase(text)
    text = remove_extra_whitespace(text)
    text = remove_stopwords(text)
    text = lemmatize_text(text)
    text = remove_extra_whitespace(text)
    return text


# ── TEST IT ──────────────────────────────────
test = "We are looking for an Excellent Python Developer with 3+ years of experience in Django & REST APIs!"

print("ORIGINAL :", test)
print("CLEANED  :", clean_text(test))

ORIGINAL : We are looking for an Excellent Python Developer with 3+ years of experience in Django & REST APIs!
CLEANED  : python developer django rest apis


In [7]:
# ── TAKE A SAMPLE ────────────────────────────
# 10,000 rows is enough for our project
# random_state=42 means we get the same sample every time

df = df.sample(n=10000, random_state=42).reset_index(drop=True)

print(f"✅ Using a sample of {len(df)} rows")
print(f"📊 Shape: {df.shape}")

✅ Using a sample of 10000 rows
📊 Shape: (10000, 12)


In [8]:
# ── APPLY CLEANING ───────────────────────────
# We clean 3 most important columns:
# 1. description → main job text
# 2. skills      → skills listed in job post
# 3. title       → job title

print("🧹 Cleaning data... this may take a few minutes\n")

tqdm.pandas()  # Enables progress bar for pandas

# Clean description column
print("Step 1/3 → Cleaning descriptions...")
df['description_clean'] = df['description'].progress_apply(clean_text)

# Clean skills column
print("\nStep 2/3 → Cleaning skills...")
df['skills_clean'] = df['skills'].progress_apply(clean_text)

# Clean title column
print("\nStep 3/3 → Cleaning titles...")
df['title_clean'] = df['title'].progress_apply(
    lambda x: to_lowercase(remove_extra_whitespace(str(x)))
)

print("\n✅ Cleaning complete!")
print(f"📊 Shape: {df.shape}")

🧹 Cleaning data... this may take a few minutes

Step 1/3 → Cleaning descriptions...


  0%|          | 0/10000 [00:00<?, ?it/s]


Step 2/3 → Cleaning skills...


  0%|          | 0/10000 [00:00<?, ?it/s]


Step 3/3 → Cleaning titles...


  0%|          | 0/10000 [00:00<?, ?it/s]


✅ Cleaning complete!
📊 Shape: (10000, 15)


In [9]:
# ── VERIFY CLEANING ──────────────────────────
# Compare original vs cleaned text

print("📋 BEFORE vs AFTER CLEANING:")
print("=" * 60)

for i in range(3):
    print(f"\n🔹 Row {i+1}:")
    print(f"TITLE     : {df['title'].iloc[i]}")
    print(f"ORIGINAL  : {str(df['description'].iloc[i])[:150]}...")
    print(f"CLEANED   : {str(df['description_clean'].iloc[i])[:150]}...")
    print(f"SKILLS    : {df['skills'].iloc[i]}")
    print(f"SK CLEAN  : {df['skills_clean'].iloc[i]}")
    print("-" * 60)

📋 BEFORE vs AFTER CLEANING:

🔹 Row 1:
TITLE     : Procurement Manager
ORIGINAL  : Promote diversity and inclusion in the supply chain, manage supplier diversity programs, and assess supplier performance....
CLEANED   : promote diversity inclusion supply chain manage supplier diversity program assess supplier performance...
SKILLS    : Supplier diversity programs Diversity and inclusion initiatives Supplier assessment and certification Data collection and reporting Vendor outreach and engagement Strategic planning Communication skills Relationship building Attention to diversity and inclusion principles
SK CLEAN  : supplier diversity program diversity inclusion initiative supplier assessment certification data collection reporting vendor outreach engagement strategic planning communication skill relationship building attention diversity inclusion principle
------------------------------------------------------------

🔹 Row 2:
TITLE     : Architectural Designer
ORIGINAL  : Architectural 

In [10]:
# ── SAVE CLEANED DATA ────────────────────────

save_path = f'{PROCESSED_DATA_PATH}/jobs_cleaned.csv'

df.to_csv(save_path, index=False)

print(f"✅ Cleaned data saved!")
print(f"📁 Location : {save_path}")
print(f"📊 Rows     : {len(df)}")
print(f"📊 Columns  : {len(df.columns)}")
print(f"\n🎉 Notebook 2 Complete!")
print(f"👉 Next step → Open Notebook 3: Skill Extraction")

✅ Cleaned data saved!
📁 Location : D:/Sinali/projects/job_market_intelligence_platform/data/processed/jobs_cleaned.csv
📊 Rows     : 10000
📊 Columns  : 15

🎉 Notebook 2 Complete!
👉 Next step → Open Notebook 3: Skill Extraction
